In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../Datasets/dataset.csv')

cols_to_drop = ["Unnamed: 0", "track_id", "mode"]
cleaned_df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
cleaned_df = cleaned_df.dropna()

numeric_cols = cleaned_df.select_dtypes(include=[np.number]).columns

Q1 = cleaned_df[numeric_cols].quantile(0.25)
Q3 = cleaned_df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

df_clean = cleaned_df[~((cleaned_df[numeric_cols] < (Q1 - 1.5 * IQR)) |
                (cleaned_df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]
df_clean.reset_index(drop=True, inplace=True)

# Ensure df_clean from earlier steps is available and has no missing values in feature columns
assert 'df_clean' in globals(), "df_clean must be defined earlier in the notebook."

# Full audio feature set (same as earlier feature_cols, excluding popularity)
full_feature_cols = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness",
    "valence", "tempo", "duration_ms"
]

# Selected feature subset (from earlier feature selection work)
selected_feature_cols = [
    "danceability", "energy", "loudness", "acousticness",
    "instrumentalness", "valence", "tempo", "duration_ms"
]


def evaluate_knn_vs_random(df, feature_cols, k=10, n_samples=1000, random_state=42):
    """Evaluate KNN (cosine distance) vs random baseline using cosine similarity.

    Returns a dict with average similarities and improvement.
    """
    rng = np.random.default_rng(random_state)

    # Drop rows with missing values in the selected features just in case
    df_feat = df.dropna(subset=feature_cols).reset_index(drop=True)

    X = df_feat[feature_cols].values

    # 1) Scale/normalize features (for the paper: StandardScaler)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 2) KNN model with cosine distance (distance = 1 - cosine_similarity)
    knn = NearestNeighbors(metric="cosine", n_neighbors=k + 1)  # +1 to include the song itself
    knn.fit(X_scaled)

    n = X_scaled.shape[0]
    n_eval = min(n_samples, n)
    eval_indices = rng.choice(n, size=n_eval, replace=False)

    knn_sims = []
    random_sims = []

    for idx in eval_indices:
        query_vec = X_scaled[idx].reshape(1, -1)

        # --- KNN recommendations ---
        distances, indices = knn.kneighbors(query_vec, n_neighbors=k + 1)
        neighbor_indices = [i for i in indices[0] if i != idx][:k]
        neighbor_distances = distances[0][1 : 1 + len(neighbor_indices)]
        # cosine similarity = 1 - cosine distance
        knn_cos_sims = 1.0 - neighbor_distances
        knn_sims.extend(knn_cos_sims.tolist())

        # --- Random baseline recommendations ---
        # Sample k random distinct indices different from the query index
        candidates = np.setdiff1d(np.arange(n), np.array([idx]))
        rand_neighbors = rng.choice(candidates, size=k, replace=False)

        rand_vecs = X_scaled[rand_neighbors]
        cos_mat = cosine_similarity(query_vec, rand_vecs)[0]
        random_sims.extend(cos_mat.tolist())

    avg_knn = float(np.mean(knn_sims))
    avg_random = float(np.mean(random_sims))
    improvement = avg_knn - avg_random

    return {
        "avg_knn": avg_knn,
        "avg_random": avg_random,
        "improvement": improvement,
    }


# ---- Hyperparameter tuning for k using selected features ----
ks_to_try = [5, 10, 20]
results_by_k = {}

print("Tuning k using selected features (cosine similarity)...")
for k in ks_to_try:
    metrics_k = evaluate_knn_vs_random(df_clean, selected_feature_cols, k=k, n_samples=1000, random_state=42)
    results_by_k[k] = metrics_k
    print(f"k = {k}: avg KNN similarity = {metrics_k['avg_knn']:.4f}, "
          f"baseline = {metrics_k['avg_random']:.4f}, improvement = {metrics_k['improvement']:.4f}")

best_k = max(results_by_k, key=lambda kk: results_by_k[kk]["avg_knn"])
print(f"\nBest k based on average cosine similarity (selected features): k = {best_k}")

# ---- Final evaluation with best k (selected features) ----
final_selected = evaluate_knn_vs_random(df_clean, selected_feature_cols, k=best_k, n_samples=2000, random_state=123)
print("\n=== Final Evaluation: Selected Feature Subset ===")
print(f"k = {best_k}")
print(f"Average cosine similarity of KNN recommendations: {final_selected['avg_knn']:.4f}")
print(f"Average cosine similarity of baseline (random) recommendations: {final_selected['avg_random']:.4f}")
print(f"Improvement over baseline: {final_selected['improvement']:.4f}")

# ---- Comparison: Full feature set vs selected subset ----
final_full = evaluate_knn_vs_random(df_clean, full_feature_cols, k=best_k, n_samples=2000, random_state=123)

print("\n=== Feature Selection Impact (using k from tuning above) ===")
print(f"Full feature set ({len(full_feature_cols)} features) - avg KNN cosine similarity: {final_full['avg_knn']:.4f}")
print(f"Selected subset ({len(selected_feature_cols)} features) - avg KNN cosine similarity: {final_selected['avg_knn']:.4f}")
print("\nUse these printed values to fill in:")
print("- Normalization method: StandardScaler")
print("- Feature selection method: KNN-based subset search / prior feature selection")
print("- Final k:", best_k)
print("- Average cosine similarity of KNN recommendations (selected features)")
print("- Average cosine similarity of baseline (random) recommendations")
print("- Improvement over baseline")
print("- Full vs. selected feature set cosine similarities for the feature-selection section of the paper.")

Tuning k using selected features (cosine similarity)...
k = 5: avg KNN similarity = 0.9813, baseline = 0.0066, improvement = 0.9748
k = 10: avg KNN similarity = 0.9756, baseline = 0.0052, improvement = 0.9704
k = 20: avg KNN similarity = 0.9681, baseline = 0.0067, improvement = 0.9614

Best k based on average cosine similarity (selected features): k = 5

=== Final Evaluation: Selected Feature Subset ===
k = 5
Average cosine similarity of KNN recommendations: 0.9810
Average cosine similarity of baseline (random) recommendations: 0.0126
Improvement over baseline: 0.9683
